# 星图映射

A股公司->从简介提取主营产品/业务->所属骨干；市值（分级？）；
1. 如果已经在图里了，gpt从主营产品/业务到板块名，对比已经在的骨干名
2. 主营产品和星图产品的相关度打分


已经定出关键词，从关键词映射到字不一样的星图词




In [1]:
import requests,json,re
from functools import wraps
from datetime import datetime
import sys
sys.path.append('E:\wangzhilin\QuantumGalaxy')
import re


from QGI.feishu import *
from QGI.mysql import MYSQL
from QGI.neoapi import Neo4j
MYSQL_HOST='localhost'
MYSQL_USER='Local_Editor'
MYSQL_PASSWORD='QuantumGalaxy'
NOVA_URI = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
NOVA_USER = "neo4j"
NOVA_PASSWORD = "QuantumGalaxy"
ATLAS_URI = "neo4j+ssc://08ef0a79.databases.neo4j.io"
ATLAS_USER = "QG_Editor"
ATLAS_PASSWORD = "editor"
def get_mysql(database='qgdbs'):

    
    #database='qgdbs'
    mysql=MYSQL(MYSQL_HOST,MYSQL_USER,MYSQL_PASSWORD,database)

    return mysql
def get_neo(database='nova'):
    if database=='nova':
        neo=Neo4j(NOVA_URI,NOVA_USER,NOVA_PASSWORD)
    else:
        neo=Neo4j(ATLAS_URI,ATLAS_USER,ATLAS_PASSWORD)
    return neo


In [2]:

def handle_res(res_func):
    @wraps(res_func)
    def wrapper(*args,**kwargs):
        
        #res={code=0,res=something}
        res=res_func(*args,**kwargs)
        if  res.status_code==200:
            if res.json().get('code')==0 :
                
                
                #code=0,tokens,res
                return res.json()['res']
            else:
                return {'code':-1,'msg':'没有您想要的结果'}
        else:
            return {'code':-2,'msg':'后台gpt服务器返回出错：%s'%res.status_code}
    return wrapper
@handle_res
def base_gpt(message):
    
    r=requests.post('http://43.153.23.232:81/chat',json=json.dumps({'message':message}))
    return r

@handle_res
def cust_gpt(messages):        
    r=requests.post('http://43.153.23.232:81/cust_chat',json=json.dumps({'messages':messages}))
    return r

@handle_res
def compltion(prompt):
    r=requests.post('http://43.153.23.232:81/completion',json=json.dumps({'prompt':prompt}))
    return r

def prod_extraction(brief):
    message=f'从下面描述中，找到公司的主营产品或业务。只回答产品/业务名\n'+brief
    r=base_gpt(message)
    return r
def guess_backbone(prod):
    messages=[{"role":'system',"content":"完成相似度选择任务。从下面的关键词表中选出与用户输入最相关的五个，然后再从选出的五个中选出与用户输入最相关的一个词。关键词表：{航空航天\n煤炭\n畜牧业\n旅游\n地产\n石油化工\n石油天然气\n钠离子电池\n半导体\n工业制造\n陆运\n海运\n电力储能\n教育\n钢铁\n锂离子电池\n风电\n消费电子\n汽车\n种植\n通信\n电影\n医药\n光伏\n互联网云计算\n纺织业\n美妆\n食品饮料\n银行\n贵金属\n}"},
        {'role':'user','content':f"{prod}与哪个关键词最相关？只回答关键词，用[]框出"}]
    r=cust_gpt(messages)
    return r
def similarity(prod_group):
    message=f'对[{prod_group[0]}]，和[{prod_group[1]}]这两组产品/业务的相关性进行评价，等级包括，几乎一致；比较相关；有些交集；毫无关系。只用回答等级'
    r=base_gpt(message)
    return r

In [ ]:
similarity(["'户外鞋靴', '运动休闲鞋'","运动鞋履的开发设计、生产与销售"])

In [3]:
s='航空航天\n煤炭\n畜牧业\n旅游\n地产\n石油化工\n石油天然气\n钠离子电池\n半导体\n工业制造\n陆运\n海运\n电力储能\n教育\n钢铁\n锂离子电池\n风电\n消费电子\n汽车\n种植\n通信\n电影\n医药\n光伏\n互联网云计算\n纺织业\n美妆\n食品饮料\n银行\n贵金属'
backbones=s.split('\n')
backbones

['航空航天',
 '煤炭',
 '畜牧业',
 '旅游',
 '地产',
 '石油化工',
 '石油天然气',
 '钠离子电池',
 '半导体',
 '工业制造',
 '陆运',
 '海运',
 '电力储能',
 '教育',
 '钢铁',
 '锂离子电池',
 '风电',
 '消费电子',
 '汽车',
 '种植',
 '通信',
 '电影',
 '医药',
 '光伏',
 '互联网云计算',
 '纺织业',
 '美妆',
 '食品饮料',
 '银行',
 '贵金属']

In [4]:
len(backbones)

30

In [5]:
def guess_backbone(prod):
    #messages=[{"role":'system',"content":"给出用户提供产品所属的产业分类。从下面的关键词表中选出与用户输入最相关的五个，然后再从选出的五个中选出与用户输入最相关的一个词。关键词表：{航空航天\n煤炭\n畜牧业\n旅游\n地产\n石油化工\n石油天然气\n钠离子电池\n半导体\n工业制造\n陆运\n海运\n电力储能\n教育\n钢铁\n锂离子电池\n风电\n消费电子\n汽车\n种植\n通信\n电影\n医药\n光伏\n互联网云计算\n纺织业\n美妆\n食品饮料\n银行\n贵金属\n}"},
    #    {'role':'user','content':f"{prod}与哪个关键词最相关？只回答关键词，用[]框出"}]
    messages=[{"role":'system',"content":"给出用户提供产品所属的产业分类class。然后，从下面的板块列表中选出与该class最接近的一个板块名，用[]框出该板块名，注意必须从列表中选择。板块列表：{航空航天\n煤炭\n畜牧业\n旅游\n地产\n石油化工\n石油天然气\n钠离子电池\n半导体\n工业制造\n陆运\n海运\n电力储能\n教育\n钢铁\n锂离子电池\n风电\n消费电子\n汽车\n种植\n通信\n电影\n医药\n光伏\n互联网云计算\n纺织业\n美妆\n食品饮料\n银行\n贵金属\n}"},
    #messages=[{"role":'system',"content":"Provide the industry classification class to which the user's product belongs. Then, select the keyword from the following keyword table that is closest in meaning to the classification, and enclose the keyword in square brackets. Keyword table:{航空航天\n煤炭\n畜牧业\n旅游\n地产\n石油化工\n石油天然气\n钠离子电池\n半导体\n工业制造\n陆运\n海运\n电力储能\n教育\n钢铁\n锂离子电池\n风电\n消费电子\n汽车\n种植\n通信\n电影\n医药\n光伏\n互联网云计算\n纺织业\n美妆\n食品饮料\n银行\n贵金属\n}"},
        {'role':'user','content':f"{prod}"}]
    r=cust_gpt(messages)
    return r
r=guess_backbone("数据服务、线上广告服务、电子产品之企业对企业、企业对客户零售业务、企业对企业交易平台、防伪产品和服务、供应链管理服务、营销服务、融资服务")
r


'[互联网云计算]'

In [ ]:
match=re.findall('\[(.*?)\]',r)

In [ ]:
match[0]

In [ ]:
re.findall('.*/[.*?/]',r)

In [ ]:
neo.close()

In [ ]:
mysql.close()

In [6]:
neo=get_neo()
mysql=get_mysql()

In [ ]:
cypher='match (c:Company)-[:produce]->(p:Product) where c.code is not null and c.code <>"无" return c.code as code,collect(p.name) as prod,c.backbone as backbone limit 100'
records=neo.read_query(cypher)
records

In [ ]:
data=[]
for r in records:
    raw=(r.get('code'),str(r.get('prod'))[1:-1],r.get('backbone'))
    data.append(raw)


In [7]:
mysql=get_mysql()
data=mysql.read_query('select code,sec_name,briefing from ticker_brief')
data

(('000001.SZ',
  '平安银行',
  '公司是深圳发展银行股份有限公司以吸收合并原平安银行股份有限公司的方式完成两行整合并更名后的银行，是全国性股份制商业银行。公司自2016年开始全面向零售银行转型，全力打造智能化零售银行，大力推进精品公司银行双轻化.综合金融是公司的特色优势，公司依托庞大的个人客户基础、强大的品牌影响力、广泛的分销网络和完备的综合金融产品体系，为全面推进零售转型、打造领先的智能化零售银行提供了强大的核心竞争力支持。在对公业务领域，公司利用大数据技术打造领先的中小企业数据征信平台，利用先进的IT系统打造橙e网、行E通、黄金银行等传统业务的新优势，提升对公司客户的数字化服务能力。'),
 ('000002.SZ',
  '万科A',
  '公司成立于1984年，1988年进入房地产行业，经过三十余年的发展，已成为国内领先的城市配套服务商，公司业务聚焦全国经济最具活力的三大经济圈及中西部重点城市。2014年万科第四个十年发展规划，已经把“三好住宅供应商”的定位延展为“城市配套服务商”。2018年万科将这一定位进一步迭代升级为“城乡建设与生活服务商”，并具体细化为四个角色：美好生活场景师，实体经济生力军，创新探索试验田，和谐生态建设者。2017年，深圳地铁集团成为本集团第一大股东，始终支持万科的混合所有制结构，支持万科城市配套服务商战略和事业合伙人机制，支持万科管理团队按照既定战略目标，实施运营和管理，支持深化“轨道+物业”发展模式。公司核心业务包括住宅开发、物业服务、长租公寓。2018年，公司将自身定位进一步迭代升级为“城乡建设与生活服务商”，所搭建的生态体系已初具规模，在巩固住宅开发和物业服务固有优势的基础上，业务已延伸至商业开发和运营、物流仓储服务、租赁住宅、产业城镇、冰雪度假、养老、教育等领域，为更好的服务人民美好生活需要、实现可持续发展奠定了良好基础。'),
 ('000004.SZ',
  'ST国华',
  '公司创立于1981年，1990年12月1日起在深圳证券交易所挂牌交易，是深交所首家挂牌的上市公司。上市之初，公司主营业务为汽车货运及旅客运输，兼营汽车修理、汽车配件零售等。2000年、2001年期间，公司完成全面资产重组，调整主营业务方向，剥离了原上市公司体内所有资产及负债，并注入了全新的生物指纹识别和生物制药产业。20

In [8]:
brief_dict={}
for row in data:
    brief_dict[row[0]]=row[2]
brief_dict

{'000001.SZ': '公司是深圳发展银行股份有限公司以吸收合并原平安银行股份有限公司的方式完成两行整合并更名后的银行，是全国性股份制商业银行。公司自2016年开始全面向零售银行转型，全力打造智能化零售银行，大力推进精品公司银行双轻化.综合金融是公司的特色优势，公司依托庞大的个人客户基础、强大的品牌影响力、广泛的分销网络和完备的综合金融产品体系，为全面推进零售转型、打造领先的智能化零售银行提供了强大的核心竞争力支持。在对公业务领域，公司利用大数据技术打造领先的中小企业数据征信平台，利用先进的IT系统打造橙e网、行E通、黄金银行等传统业务的新优势，提升对公司客户的数字化服务能力。',
 '000002.SZ': '公司成立于1984年，1988年进入房地产行业，经过三十余年的发展，已成为国内领先的城市配套服务商，公司业务聚焦全国经济最具活力的三大经济圈及中西部重点城市。2014年万科第四个十年发展规划，已经把“三好住宅供应商”的定位延展为“城市配套服务商”。2018年万科将这一定位进一步迭代升级为“城乡建设与生活服务商”，并具体细化为四个角色：美好生活场景师，实体经济生力军，创新探索试验田，和谐生态建设者。2017年，深圳地铁集团成为本集团第一大股东，始终支持万科的混合所有制结构，支持万科城市配套服务商战略和事业合伙人机制，支持万科管理团队按照既定战略目标，实施运营和管理，支持深化“轨道+物业”发展模式。公司核心业务包括住宅开发、物业服务、长租公寓。2018年，公司将自身定位进一步迭代升级为“城乡建设与生活服务商”，所搭建的生态体系已初具规模，在巩固住宅开发和物业服务固有优势的基础上，业务已延伸至商业开发和运营、物流仓储服务、租赁住宅、产业城镇、冰雪度假、养老、教育等领域，为更好的服务人民美好生活需要、实现可持续发展奠定了良好基础。',
 '000004.SZ': '公司创立于1981年，1990年12月1日起在深圳证券交易所挂牌交易，是深交所首家挂牌的上市公司。上市之初，公司主营业务为汽车货运及旅客运输，兼营汽车修理、汽车配件零售等。2000年、2001年期间，公司完成全面资产重组，调整主营业务方向，剥离了原上市公司体内所有资产及负债，并注入了全新的生物指纹识别和生物制药产业。2005年，为了优化上市公司资产质量，提高盈利能力，公司新增房地产开发与销售业务，并

In [9]:
import pandas as pd
df=pd.DataFrame(data=data,columns=['code','prod','backbone'])
df=df.set_index('code')
df['gpt_prod']=None
df['gpt_backbone']=None
df['similarity']=None
backbone_pattern=re.compile('\[(.*?)\]')
num=0
for code,r in df.iterrows():
    sql=f'select briefing from ticker_brief where code="{code}"'
    brief=mysql.read_query(sql)
    if brief:
        brief=brief[0][0]
        
    else:
        print(code+' has no brief in mysql\n')
        
        continue
    
    gpt_prod=prod_extraction(brief)
    df.loc[code,'gpt_prod']=gpt_prod
    gpt_backbone=guess_backbone(gpt_prod)
    find=backbone_pattern.findall(gpt_backbone)
    if find:
        gpt_backbone=find[0]
        df.loc[code,'gpt_backbone']=gpt_backbone
    gpt_similarity=similarity([r['prod'],gpt_prod])
    df.loc[code,'similarity']=gpt_similarity
    num+=1
    print(num)
    
df

1


KeyboardInterrupt: 

In [10]:
import pandas as pd
df=pd.DataFrame(data=data,columns=['code','name','briefing'])
df=df.set_index('code')
df['gpt_prod']=None
df['gpt_backbone']=None
#df['similarity']=None
backbone_pattern=re.compile('\[(.*?)\]')
df


,name,briefing,gpt_prod,gpt_backbone
code,,,,
000001.SZ,平安银行,公司是深圳发展银行股份有限公司以吸收合并原平安银行股份有限公司的方式完成两行整合并更名后的银...,None,None
000002.SZ,万科A,公司成立于1984年，1988年进入房地产行业，经过三十余年的发展，已成为国内领先的城市配套...,None,None
000004.SZ,ST国华,公司创立于1981年，1990年12月1日起在深圳证券交易所挂牌交易，是深交所首家挂牌的上市...,None,None
000005.SZ,ST星源,公司是中国早期大规模介入酒店式公寓服务业的公司之一，中国早期上市的公司之一。公司从事的主要业...,None,None
000006.SZ,深振业A,公司是深圳市国有资产监督管理委员会直管的国有上市公司，公司以房地产开发经营为主营业务，具备国...,None,None
...,...,...,...,...
YZCF.HK,兖州煤业股期,香港交易所是香港联合交易所有限公司、香港期货交易所有限公司和香港中央结算有限公司的控股公司。...,None,None
ZAOF.HK,HKEX众安在线财产保险股份有限公司股票期货,香港交易所是香港联合交易所有限公司、香港期货交易所有限公司和香港中央结算有限公司的控股公司。...,None,None
ZC.CZC,CZCE动力煤,郑州商品交易所成立于1990年10月12日，是经国务院批准成立的国内首家期货市场试点单位，于...,None,None


In [ ]:
df['gpt_prod']=None
df['gpt_backbone']=None
#df['similarity']=None
backbone_pattern=re.compile('\[(.*?)\]')
num=0
for code,r in df.iterrows():
    sql=f'select briefing from ticker_brief where code="{code}"'
    brief=mysql.read_query(sql)
    if brief:
        brief=brief[0][0]
        
    else:
        print(code+' has no brief in mysql\n')
        
        continue
    
    gpt_prod=prod_extraction(brief)
    df.loc[code,'gpt_prod']=gpt_prod
    gpt_backbone=guess_backbone(gpt_prod)
    find=backbone_pattern.findall(gpt_backbone)
    if find:
        gpt_backbone=find[0]
        if gpt_backbone in backbones:
            df.loc[code,'gpt_backbone']=gpt_backbone
        else:
            print(gpt_backbone)
            df.loc[code,'gpt_backbone']='未找到分类'
    #gpt_similarity=similarity([r['prod'],gpt_prod])
    #df.loc[code,'similarity']=gpt_similarity
    num+=1
    print(num)
    if num==1:
        break
    
df

In [11]:
codelist=df.index.tolist()
len(codelist)

8180

In [12]:
len(codelist)/5

1636.0

In [13]:
5%100

5

In [14]:
from tqdm import tqdm
import time

for i in tqdm(range(100)):
    time.sleep(0.1)

100%|██████████| 100/100 [00:10<00:00,  9.84it/s]


In [15]:


import threading

# 定义一个处理列表的函数
def process_list(lst):
    num=0
    for code in lst:
        #code=lst[i]
        try:
            brief=brief_dict[code]
            gpt_prod=prod_extraction(brief)
            df.loc[code,'gpt_prod']=gpt_prod
            gpt_backbone=guess_backbone(gpt_prod)
            find=backbone_pattern.findall(gpt_backbone)
            if find:
                gpt_backbone=find[0]
                if gpt_backbone in backbones:

                    df.loc[code,'gpt_backbone']=gpt_backbone
                else:

                    print('内容不在范围内：'+code+','+gpt_prod+":"+gpt_backbone)
                    df.loc[code,'gpt_backbone']='未找到分类'
            else:
                print('格式不对：'+code+','+gpt_prod+":"+gpt_backbone)
            num+=1
            if num%100==0:
                print(num)
        except:
            pass
        

# 定义一个线程类
class ListThread(threading.Thread):
    def __init__(self, lst):
        threading.Thread.__init__(self)
        self.lst = lst

    def run(self):
        # 在线程中调用处理列表的函数
        process_list(self.lst)

# 创建一个列表
codelist=df.index.tolist()

# 定义线程数
num_threads = 5

# 计算每个线程需要处理的列表长度
chunk_size = len(codelist) // num_threads

# 创建线程列表
threads = []

# 将列表分成多个块，并为每个块创建一个线程
for i in range(num_threads):
    start = i * chunk_size
    end = start + chunk_size
    if i == num_threads - 1:
        end = len(codelist)
    thread = ListThread(codelist[start:end])
    threads.append(thread)

# 启动所有线程
for thread in threads:
    thread.start()

# 等待所有线程完成
for thread in threads:
    thread.join()

格式不对：2280.HK,数据服务、线上广告服务、电子产品之企业对企业、企业对客户零售业务、企业对企业交易平台、防伪产品和服务、供应链管理服务、营销服务、融资服务。:互联网云计算
内容不在范围内：0122.HK,鳄鱼恤、鳄鱼恤女装、鳄鱼仔、Lacoste:时尚服装
内容不在范围内：0128.HK,主营业务为投资控股及证券买卖。:证券
内容不在范围内：000010.SZ,园林绿化工程施工、园林景观设计、园林养护及绿化苗木种植。:园林
内容不在范围内：600055.SH,医用X射线诊断设备、磁共振成像设备、齿科诊断和治疗设备、血液健康产品:"医药"
内容不在范围内：600057.SH,公司主营产品/业务为商品采购供应及综合物流服务、物流园区平台开发运营。:商品采购供应
格式不对：2297.HK,caFFR系统和caIMR系统:由于没有明确的上下文和相关信息，无法确定caFFR系统和caIMR系统的具体含义和指代。请提供更多背景信息或上下文，以便更好地回答您的问题。
格式不对：605377.SH,可印刷装饰原纸和素色装饰原纸:工业制造
内容不在范围内：2306.HK,艺人管理、音乐IP制作及运营及泛娱乐业务:音乐
格式不对：600058.SH,冶金原材料和钢材的国内外贸易。:根据题目，我们可以将该问题分成两个部分来回答。

1. 冶金原材料的国内外贸易：

我国的冶金原材料主要包括铁矿石、铜矿石、铝土矿等。根据中国海关统计，2019年我国进口的铁矿石总量为10.39亿吨，同比增长0.5%。其中，澳大利亚、巴西和南非是我国的三大铁矿石进口国。而我国的铜矿石主要进口自智利、秘鲁、澳大利亚等国家。此外，我国的铝土矿主要来自澳大利亚、几内亚等国家。

2. 钢材的国内外贸易：

我国是全球最大的钢铁生产和消费国之一。根据中国海关统计，2019年我国的钢材出口量为6426万吨，同比下降7.3%，而钢材进口量为12230万吨，同比下降7.8%。我国的钢材进口主要来自于日本、韩国、东南亚等国家和地区，而钢材出口则主要流向东南亚、美国、欧洲等地。受到新冠疫情和国际贸易摩擦等因素的影响，我国的钢材进出口总量在近两年出现了一定程度的下降。
格式不对：2309.HK,彩票系统及在线付款系统服务解决方案。:该产品属于互联网云计算板块。
格式不对：2310.HK,电子组件制造及销售、物业投资、酒店

In [16]:
df.dropna(subset='gpt_prod').to_csv('E:\wangzhilin\QuantumGalaxy\server_openai\gptbackbone1.csv')


In [22]:
'1'=='1'

True

In [25]:
df[(df['gpt_backbone'].isnull()) | (df['gpt_backbone'] == '未找到分类')]

,name,briefing,gpt_prod,gpt_backbone
code,,,,
000010.SZ,美丽生态,公司是中国最早的上市公司之一，公司主营业务为园林绿化，主要从事园林绿化工程施工、园林景观设计...,园林绿化工程施工、园林景观设计、园林养护及绿化苗木种植。,未找到分类
000020.SZ,深华发A,公司是深圳证券交易所早期上市的公司之一，是全国外商投资双优企业。公司经营业务主要分为液晶显示...,液晶显示器整机业务、注塑件生产业务、保丽龙(轻型材料包装)生产业务、物业租赁业务。,None
000023.SZ,深天地A,公司是全国第一家以商品混凝土为主业的上市公司，主要业务涉及商品混凝土、房地产开发及物业管理等...,商品混凝土、预拌砂浆,未找到分类
000039.SZ,中集集团,公司是世界领先的物流装备和能源装备供应商，总部位于中国深圳。公司致力于在如下主要业务领域：集...,集装箱、道路运输车辆、能源化工及食品装备、海洋工程、物流服务、空港设备。,None
000040.SZ,东旭蓝天,公司是国内A股主板上市公司，主营业务包括新能源和生态环保。公司致力于成为国内领先的环保新能源...,智慧能源、生态综合治理、水环境修复及运营、土壤及矿山修复、危废/固废处理、新材料、安轩智能运...,None
...,...,...,...,...
WR.SHF,SHFE线材,上海期货交易所是依照有关法规设立的，履行有关法规规定的职能，按照其章程实行自律性管理的法人，...,上海期货交易所的主营业务是期货交易。,None
WXBF.HK,HKEX药明生物技术股期,香港交易所是香港联合交易所有限公司、香港期货交易所有限公司和香港中央结算有限公司的控股公司。...,香港交易所的主营业务为运营香港中央证券及衍生产品市场，提供交易、结算、存管、代理人和资讯服务。,未找到分类
YZCF.HK,兖州煤业股期,香港交易所是香港联合交易所有限公司、香港期货交易所有限公司和香港中央结算有限公司的控股公司。...,香港交易所的主营业务为运营香港中央证券及衍生产品市场，提供交易、结算、存管、代理人和资讯服务。,未找到分类


In [ ]:
df['name']=None
for code,r in df.iterrows():
    sql=f'select sec_name from ticker_brief where code="{code}"'
    name=mysql.read_query(sql)
    if name:
        df.loc[code,'name']=name[0][0]

In [ ]:
df.to_csv('E:\wangzhilin\QuantumGalaxy\server_openai\gptbackbone.csv')

In [ ]:
records[0].get('code',None)

In [ ]:
for r in records:
    code=r.get('code')
    prod=r.get('prod')
    backbone=r.get('backbone')
    sql=f'select briefing from ticker_brief where code="{code}"'
    brief=r=mysql.read_query(sql)
    if brief:
        breif=r[0][0]
    else:
        continue
    gpt_prod=prod_extraction(brief)
    


In [ ]:
code='300979.SZ'
sql=f'select briefing from ticker_brief where code="{code}"'
r=mysql.read_query(sql)

In [ ]:
r[0][0]

In [ ]:
cypher='match (c:Company)-[:produce]->(p:Product) where c.code ="603083.SH"  return c.code,collect(p.name) limit 2'
r=neo.read_query(cypher)
r